# 01 — Data Extraction & Profiling

This is the first step of our NYC Airbnb analysis. We're using the **Inside Airbnb** dataset for New York City (2019).

**Goal:** Load the raw data, see what's inside, and figure out what needs cleaning.

### Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Settings for better display
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style="whitegrid")

# Path to our data
data_path = Path('../data/raw/AB_NYC_2019.csv')
if not data_path.exists():
    data_path = Path('data/raw/AB_NYC_2019.csv') # fallback if run from root

### Loading the data

In [2]:
df = pd.read_csv(data_path)
print(f"Loaded {len(df)} rows and {len(df.columns)} columns.")
df.head()

Loaded 48895 rows and 16 columns.


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.65,-73.97,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75,-73.98,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.81,-73.94,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.69,-73.96,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.80,-73.94,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


### Initial Data Profiling
Let's check data types and look for missing values.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  int64  
 1   name                            48879 non-null  object 
 2   host_id                         48895 non-null  int64  
 3   host_name                       48874 non-null  object 
 4   neighbourhood_group             48895 non-null  object 
 5   neighbourhood                   48895 non-null  object 
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type                       48895 non-null  object 
 9   price                           48895 non-null  int64  
 10  minimum_nights                  48895 non-null  int64  
 11  number_of_reviews               48895 non-null  int64  
 12  last_review                     

In [4]:
# Checking for nulls
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df)) * 100
null_summary = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
null_summary[null_summary['null_count'] > 0].sort_values('null_count', ascending=False)

,null_count,null_pct
last_review,10052,20.56
reviews_per_month,10052,20.56
host_name,21,0.04
name,16,0.03


**Observations:**
- `reviews_per_month` and `last_review` have a lot of missing values (~20%). This probably means these listings haven't been reviewed yet.
- `name` and `host_name` have a few missing entries, but not many.

### Exploring Main Categories

In [5]:
print("Neighbourhood Groups:")
print(df['neighbourhood_group'].value_counts())

print("\nRoom Types:")
print(df['room_type'].value_counts())

Neighbourhood Groups:
neighbourhood_group
Manhattan        21661
Brooklyn         20104
Queens            5666
Bronx             1091
Staten Island      373
Name: count, dtype: int64

Room Types:
room_type
Entire home/apt    25409
Private room       22326
Shared room         1160
Name: count, dtype: int64


### Checking Numeric Ranges
We need to see if there are any weird values like $0 prices or extreme minimum nights.

In [6]:
df[['price', 'minimum_nights', 'number_of_reviews', 'availability_365']].describe()

,price,minimum_nights,number_of_reviews,availability_365
count,48895.00,48895.00,48895.00,48895.00
mean,152.72,7.03,23.27,112.78
std,240.15,20.51,44.55,131.62
min,0.00,1.00,0.00,0.00
25%,69.00,1.00,1.00,0.00
50%,106.00,3.00,5.00,45.00
75%,175.00,5.00,24.00,227.00
max,10000.00,1250.00,629.00,365.00


In [7]:
# Let's look closer at the anomalies
print(f"Listings with price 0: {len(df[df['price'] == 0])}")
print(f"Listings with min nights > 365: {len(df[df['minimum_nights'] > 365])}")
print(f"Listings with 0 availability: {len(df[df['availability_365'] == 0])}")

Listings with price 0: 11
Listings with min nights > 365: 14
Listings with 0 availability: 17533


### Summary of Findings
Here's what we found that needs fixing in the next notebook:
1. **Missing Values**: Fill `reviews_per_month` with 0 and `name`/`host_name` with placeholders.
2. **Zero Prices**: 11 listings have a price of $0. We should remove these.
3. **Price Outliers**: Max price is $10,000, which is likely an outlier. We'll need to cap this.
4. **Minimum Nights**: Some values are over 365 days, which seems like a mistake for a rental platform.